# Refatoração em Python
## Cheiros, Refatorações e o Ciclo Red–Green–Refactor

**Prof. Dr. André Yoshiaki Kashiwabara** — DCOMP / UFS  
`andreyoshiaki@dcomp.ufs.br`

<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook acompanha os slides da aula. A ideia é que você **rode**, **quebre** e **refatore** o código você mesmo — passando pelo ciclo *Red → Green → Refactor* em cada seção.

### Roteiro
1. Parte I — **Problema**: maus cheiros de código.
2. Parte II — **Solução**: catálogo de refatorações.
3. Parte III — **Disciplina**: ciclo Red–Green–Refactor com `pytest` e PEP 8.

## Setup — rodando `pytest` dentro do Colab

Usaremos `ipytest` para executar testes diretamente nas células do notebook.

In [ ]:
!pip install -q ipytest ruff black
import ipytest
ipytest.autoconfig()

---
# Parte I — Problema: Maus Cheiros de Código

> Um *code smell* é uma estrutura no código que **sugere** — mas não prova — a necessidade de refatoração. — Beck & Fowler

As quatro famílias de Fowler:

| Família | Exemplos |
|---|---|
| **Bloaters** | Long Method, Large Class, Long Parameter List |
| **OO Abusers** | Switch Statements, Temporary Field, Refused Bequest |
| **Change Preventers** | Divergent Change, Shotgun Surgery, Feature Envy |
| **Dispensables** | Duplicated Code, Dead Code, Speculative Generality |

## Smell 1 — Long Method
Função longa, com várias responsabilidades misturadas.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Item:
    preco: float
    qtd: int

@dataclass
class Cliente:
    nome: str
    email: str

@dataclass
class Pedido:
    id: int
    cliente: Cliente
    itens: List[Item] = field(default_factory=list)


def processar_pedido(p):
    if not p.cliente: raise ValueError('sem cliente')
    if not p.itens:   raise ValueError('sem itens')
    total = 0
    for i in p.itens: total += i.preco * i.qtd
    if total > 1000: total *= 0.9
    msg = f'Pedido {p.id}: R$ {total}'
    print('[email enviado]', p.cliente.email, '->', msg)
    return total

pedido = Pedido(1, Cliente('Ana', 'ana@ex.com'), [Item(500, 1), Item(700, 1)])
processar_pedido(pedido)

### Smell 2 — Duplicated Code

In [ ]:
def preco_a(x): return x if x > 0 else -x
def preco_b(y): return y if y > 0 else -y
def preco_c(z): return z if z > 0 else -z

print(preco_a(-3), preco_b(-3), preco_c(-3))  # 3 3 3 — três lugares, uma ideia

### Smell 3 — Switch / cadeia `if-elif` sobre *strings mágicas*

In [ ]:
def calcular_frete(tipo, peso):
    if tipo == 'NORMAL':     return peso * 5.0
    elif tipo == 'EXPRESSO': return peso * 10.0
    elif tipo == 'SEDEX':    return peso * 15.0
    else: raise ValueError(tipo)

calcular_frete('SEDEX', 2)

> **Exercício:** liste, em uma célula de texto abaixo, ao menos um cheiro adicional que você consegue identificar em cada um dos três exemplos acima.

---
# Parte II — Solução: Refatoração

> Refatorar é reestruturar código existente alterando sua estrutura interna **sem mudar seu comportamento externo observável**. — Fowler (2018)

Catálogo mínimo:
- **Extract Function**
- **Rename**
- **Inline**
- **Introduce Parameter Object**
- **Replace Conditional with Lookup / Polymorphism**

## Extract Function — antes

In [ ]:
@dataclass
class Lancamento:
    descricao: str
    valor: float

def imprimir_extrato(cliente, lancamentos):
    print(f'Cliente: {cliente.nome}')
    print('-' * 30)
    total = 0
    for l in lancamentos:
        total += l.valor
    print(f'Total: R$ {total:.2f}')

imprimir_extrato(Cliente('Ana', 'ana@ex.com'),
                 [Lancamento('a', 10), Lancamento('b', 20)])

## Extract Function — depois

In [ ]:
def calcular_total(lancamentos):
    return sum(l.valor for l in lancamentos)

def imprimir_extrato(cliente, lancamentos):
    print(f'Cliente: {cliente.nome}')
    print('-' * 30)
    total = calcular_total(lancamentos)
    print(f'Total: R$ {total:.2f}')

imprimir_extrato(Cliente('Ana', 'ana@ex.com'),
                 [Lancamento('a', 10), Lancamento('b', 20)])

## Replace Conditional with Lookup

In [ ]:
TAXA_FRETE = {'NORMAL': 5.0, 'EXPRESSO': 10.0, 'SEDEX': 15.0}

def calcular_frete(tipo, peso):
    if tipo not in TAXA_FRETE:
        raise ValueError(tipo)
    return peso * TAXA_FRETE[tipo]

calcular_frete('SEDEX', 2)

## Estudo de caso — refatorando do cheiroso ao limpo

**Ponto de partida (cheirando mal):**

In [ ]:
def calc(itens, t):
    s = 0
    for i in itens:
        s = s + i[0] * i[1]
    if t == 'vip':
        s = s * 0.9
    elif t == 'func':
        s = s * 0.8
    return s

calc([(10, 2), (5, 3)], 'vip')

**Depois das refatorações** (Rename + Extract Function + Replace Conditional with Lookup):

In [ ]:
DESCONTO = {'vip': 0.10, 'func': 0.20}

@dataclass
class ItemPedido:
    preco: float
    qtd: int

def subtotal(itens):
    return sum(item.preco * item.qtd for item in itens)

def calcular_total(itens, perfil='comum'):
    return subtotal(itens) * (1 - DESCONTO.get(perfil, 0))

itens = [ItemPedido(10, 2), ItemPedido(5, 3)]
calcular_total(itens, 'vip')

---
# Parte III — Disciplina: Red–Green–Refactor com `pytest`

1. **Red** — escreva um teste que falha. Ele define o que o código *deve* fazer.
2. **Green** — faça-o passar do jeito mais simples.
3. **Refactor** — com a rede verde, melhore o design sem mudar comportamento.

> Os dois chapéus de Kent Beck: **ou** você muda *comportamento*, **ou** muda *estrutura* — nunca os dois ao mesmo tempo.

## Red — escreva o teste primeiro

In [ ]:
%%ipytest -q

# Ainda não existe — o teste deve falhar (Red).
def test_total_com_desconto():
    assert total([10, 20, 30], desconto=0.1) == 54.0

## Green — o mínimo que faz passar

In [ ]:
def total(itens, desconto):
    return sum(itens) * (1 - desconto)

In [ ]:
%%ipytest -q

def test_total_com_desconto():
    assert total([10, 20, 30], desconto=0.1) == 54.0

## Refactor — nomes e responsabilidades (testes continuam passando)

In [ ]:
def _aplicar_desconto(valor, taxa):
    return valor * (1 - taxa)

def total(itens, desconto=0.0):
    return _aplicar_desconto(sum(itens), desconto)

In [ ]:
%%ipytest -q

import pytest

def test_total_lista_vazia():
    assert total([]) == 0

def test_total_sem_desconto():
    assert total([1, 2, 3]) == 6

@pytest.mark.parametrize('entrada,desc,esperado', [
    ([10, 20, 30], 0.0, 60.0),
    ([10, 20, 30], 0.1, 54.0),
    ([100], 0.5, 50.0),
])
def test_total_parametrizado(entrada, desc, esperado):
    assert total(entrada, desc) == esperado

## Exceções esperadas — documentando o contrato

In [ ]:
class SaldoInsuficienteError(Exception):
    pass

@dataclass
class Conta:
    saldo: float
    def sacar(self, valor):
        if valor > self.saldo:
            raise SaldoInsuficienteError(valor)
        self.saldo -= valor
        return self.saldo

In [ ]:
%%ipytest -q

import pytest

def test_sacar_dentro_do_saldo():
    assert Conta(100).sacar(40) == 60

def test_sacar_mais_que_saldo_falha():
    c = Conta(saldo=100)
    with pytest.raises(SaldoInsuficienteError):
        c.sacar(150)

## PEP 8 na prática — `ruff` e `black`

**Antes** (viola convenções):

In [ ]:
codigo_feio = '''import os,sys
def CalcularMedia( numeros ):
  Total=0
  for N in numeros:Total=Total+N
  return Total/len( numeros )
'''

with open('exemplo.py', 'w') as f:
    f.write(codigo_feio)

!ruff check exemplo.py || true

In [ ]:
!black -q exemplo.py
!ruff check --fix exemplo.py || true
print('---- depois ----')
print(open('exemplo.py').read())

> **Observe:** `black` e `ruff` ajustaram indentação, espaços e quebraram o `import` múltiplo, mas **não renomearam** `CalcularMedia` → `calcular_media` nem `Total` → `total`. Renomear identificadores é uma **refatoração** (*Rename*) — depende de intenção humana e do impacto em quem chama a função. Formatadores cuidam de estética mecânica; refatorações cuidam de design.

---
# Exercício final integrador

1. Identifique os cheiros na função abaixo.
2. **Antes de mexer**, escreva testes `pytest` que capturem o comportamento atual.
3. Refatore em passos pequenos, rodando os testes a cada passo.
4. Rode `ruff`/`black` no final.

In [ ]:
def relatorio(ps):
    out = ''
    tot = 0
    for p in ps:
        sub = 0
        for it in p['itens']:
            sub = sub + it['p'] * it['q']
        if p['t'] == 'vip': sub = sub * 0.9
        elif p['t'] == 'func': sub = sub * 0.8
        out = out + p['c'] + ': ' + str(sub) + '\n'
        tot = tot + sub
    out = out + 'TOTAL: ' + str(tot)
    return out

exemplo = [
    {'c': 'Ana',  't': 'vip',  'itens': [{'p': 10, 'q': 2}, {'p': 5, 'q': 3}]},
    {'c': 'Beto', 't': 'func', 'itens': [{'p': 100, 'q': 1}]},
]
print(relatorio(exemplo))

## Leituras recomendadas
- Fowler (2018). *Refactoring*, caps. 1 e 3.
- Beck (2003). *TDD by Example*, parte I.
- Martin (2008). *Clean Code*.
- PEP 8 — <https://peps.python.org/pep-0008/>
- Catálogo de refatorações — <https://refactoring.guru>

> *“Make it work, make it right, make it fast.”* — Kent Beck